In [1]:
import pandas as pd


In [2]:
DTYPES = {
    "Invoice": str,
    "StockCode": str,
}

In [3]:
from pathlib import Path
Path.cwd()

WindowsPath('c:/Users/OMISTAJA/Documents/repos/mini-data-pipeline/notebooks')

In [4]:
df = pd.read_csv(
    "../data/combined_retail_data.csv",
    dtype=DTYPES,
    parse_dates=["InvoiceDate"]
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1044848 entries, 0 to 1044847
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1044848 non-null  str           
 1   StockCode    1044848 non-null  str           
 2   Description  1040573 non-null  str           
 3   Quantity     1044848 non-null  int64         
 4   InvoiceDate  1044848 non-null  datetime64[us]
 5   Price        1044848 non-null  float64       
 6   Customer ID  809561 non-null   float64       
 7   Country      1044848 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 63.8 MB


## Column profile vs. data contract

Rows: 1,044,848 (matches extract output)

| Column | Contract | Data | Result |
|---|---|---|---|
| Invoice | text, required | `str` (forced on read), 0 missing | OK, Format rule not yet checked |
| StockCode | text, required | `str` (forced on read), 0 missing | OK, Format rule not yet checked |
| Description | text, optional | `str`, 4,275 missing (0.4 %) | OK |
| Quantity | integer, required | `int64` (inferred), 0 missing | All values are integers. Value rules not yet checked |
| InvoiceDate | timestamp, required | `datetime64` (parsed), 0 missing | All timestamps valid |
| Price | decimal, required | `float64` (inferred), 0 missing | Value rules not yet checked |
| Customer ID | text, optional | `float64`, 235,287 missing (22.5 %) | Float due to NaN, FIX in transform. Nearly a quarter of sales has no customer |
| Country | text, optional | `str`, 0 missing | OK |

In [5]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.044848e+06,1044848,1.044848e+06,809561.000000
mean,9.993649e+00,2011-01-03 11:44:15.916841,4.590546e+00,15319.760422
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-05 11:11:00,1.250000e+00,13969.000000
50%,3.000000e+00,2010-12-08 16:34:00,2.100000e+00,15248.000000
75%,1.000000e+01,2011-07-27 13:42:00,4.130000e+00,16792.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.742185e+02,NaN,1.217042e+02,1695.812057


In [6]:
# Count of rows where Quantity is 0
(df["Quantity"] == 0).sum()


np.int64(0)

In [7]:
# Rows where Price is negative
df[df["Price"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
802921,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
802922,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [8]:
# Rows where Quantity is 80995 or -80995
df[df["Quantity"].abs() == 80995]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1043359,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
1043360,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


In [9]:
df[df["Invoice"].str.startswith("A")]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
802920,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
802921,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
802922,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


| Check | Contract | Finding | Result |
|---|---|---|---|
| Quantity = 0 | ≠ 0 → REJECT | 0 rows | OK |
| Quantity extremes | negative only on `C` invoices | ±80,995 = order `581483` cancelled 12 min later by `C581484` (£168,469.60) | OK, Cancellations must be kept, otherwise sales are overstated |
| Cancellation link | — | Cancellation invoice does not reference the original invoice number | OPEN: link can only be inferred from customer + product + quantity |
| Price < 0 | < 0 → REJECT | 5 rows, all `Adjust bad debt` |  New row type, see below |
| `A` invoices | not documented | 6 rows: `Invoice` prefix `A`, `StockCode` `B`, no customer. Net −£147,614.08 |  Undocumented. Accounting adjustments, not sales → REJECT with reason |
| InvoiceDate range | not in the future | 2009-12-01 – 2011-12-09 | OK |
| Customer ID range | 5 digits | 12346 – 18287 | OK |
| Description with comma | — | e.g. `PAPER CRAFT , LITTLE BIRDIE` | OK, Correctly quoted in CSV |